# Project: PR Review Assistant
**Brief from Paras:**
GitHub webhook on new PR. Fetches diff, summarises, checks Notion review rules,
posts structured review comment, pings reviewer on Slack.

This is a SCAFFOLD. The supervisor + graph wiring is complete; worker logic for some nodes is marked TODO.
Use Project #2 (`02_support_resolver/support_resolver.ipynb`) as your reference implementation.

## Submission checklist
- [ ] All worker TODOs filled in
- [ ] Composio actions verified against docs.composio.dev
- [ ] HITL where destructive actions occur
- [ ] Checkpoint persistence configured
- [ ] Graph diagram saved as graph.png
- [ ] README.md with architecture + example run


## 1. Setup

In [ ]:
import os, sqlite3
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY")
assert os.getenv("COMPOSIO_API_KEY"), "Sign up at composio.dev and connect required apps in their dashboard"
print("env OK")


## 2. State schema

In [ ]:
from typing import TypedDict, Annotated, Optional, Literal
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage, SystemMessage

class PrReviewState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    next_worker: str
    pr_number: int
    repo: str
    diff_text: Optional[str]
    summary: Optional[str]
    rule_violations: list[str]
    review_posted: bool


## 3. LLM and Composio toolset

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from composio_langgraph import Action, ComposioToolSet

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
toolset = ComposioToolSet()


## 4. Workers

In [ ]:
# diff_fetcher - GitHub diff fetcher
diff_fetcher_tools = toolset.get_tools(actions=[Action.GITHUB_GET_PR_DIFF])
diff_fetcher_agent = create_react_agent(llm, diff_fetcher_tools, prompt='''Fetch diff for PR. Return diff_text.''')
def diff_fetcher_node(state):
    result = diff_fetcher_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='diff_fetcher')]}


# summariser - Diff summariser (pure LLM)
# No Composio actions; pure LLM or custom logic.
# TODO: implement summariser_node(state) following the pattern in 02_support_resolver.
def summariser_node(state):
    """Summarise diff in plain English: what changed, which files, why."""
    raise NotImplementedError('TODO: implement summariser_node')


# rule_checker - Notion rules checker
rule_checker_tools = toolset.get_tools(actions=[Action.NOTION_QUERY_DATABASE])
rule_checker_agent = create_react_agent(llm, rule_checker_tools, prompt='''Pull review rules from Notion (tests required? docs required?). Compare to diff. Return violations list.''')
def rule_checker_node(state):
    result = rule_checker_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='rule_checker')]}


# commenter - PR comment poster
# No Composio actions; pure LLM or custom logic.
# TODO: implement commenter_node(state) following the pattern in 02_support_resolver.
def commenter_node(state):
    """Post structured review comment on the PR via GitHub MCP server."""
    raise NotImplementedError('TODO: implement commenter_node')


# notifier - Slack reviewer notifier
notifier_tools = toolset.get_tools(actions=[Action.SLACK_SENDS_A_MESSAGE_TO_A_SLACK_CHANNEL])
notifier_agent = create_react_agent(llm, notifier_tools, prompt='''Tag the assigned reviewer with PR link and violation summary.''')
def notifier_node(state):
    result = notifier_agent.invoke({'messages': state['messages']})
    last = result['messages'][-1]
    return {'messages': [AIMessage(content=last.content, name='notifier')]}


## 5. Supervisor + router

In [ ]:
def supervisor(state) -> dict:
    if state.get('diff_text') is None: return {'next_worker': 'diff_fetcher'}
    if state.get('summary') is None: return {'next_worker': 'summariser'}
    if not state['rule_violations'] and 'rules_checked' not in str(state['messages']):
        return {'next_worker': 'rule_checker'}
    if not state['review_posted']: return {'next_worker': 'commenter'}
    if 'reviewer notified' not in (state['messages'][-1].content.lower() if state['messages'] else ''):
        return {'next_worker': 'notifier'}
    return {'next_worker': 'DONE'}

def route(state) -> str:
    nxt = state['next_worker']
    if nxt in {'diff_fetcher', 'commenter', 'notifier', 'summariser', 'rule_checker'}:
        return nxt
    return '__end__'


## 6. Build graph + checkpoint persistence

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

g = StateGraph(globals()[[k for k in globals() if k.endswith('State') and k != 'AgentState'][0]])
g.add_node("supervisor", supervisor)
g.add_node("diff_fetcher", diff_fetcher_node)
g.add_node("summariser", summariser_node)
g.add_node("rule_checker", rule_checker_node)
g.add_node("commenter", commenter_node)
g.add_node("notifier", notifier_node)

g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", route, {
    "diff_fetcher": "diff_fetcher",
    "summariser": "summariser",
    "rule_checker": "rule_checker",
    "commenter": "commenter",
    "notifier": "notifier",
    "__end__": END,
})
g.add_edge("diff_fetcher", "supervisor")
g.add_edge("summariser", "supervisor")
g.add_edge("rule_checker", "supervisor")
g.add_edge("commenter", "supervisor")
g.add_edge("notifier", "supervisor")

conn = sqlite3.connect("08_pr_review.db", check_same_thread=False)
app = g.compile(checkpointer=SqliteSaver(conn))


## 7. Visualise (submission requirement)

In [ ]:
from IPython.display import Image, display
try:
    png = app.get_graph().draw_mermaid_png()
    Path("graph.png").write_bytes(png)
    display(Image("graph.png"))
except Exception as e:
    print("ASCII fallback:")
    print(app.get_graph().draw_ascii())


## 8. End-to-end demo

In [ ]:
config = {'configurable': {'thread_id': '08_pr_review-demo-1'}, 'recursion_limit': 30}

result = app.invoke(
    {'pr_number': 42, 'repo': 'acme/api',
        'messages': [HumanMessage(content='New PR opened')]},
    config=config,
)

print("=== FINAL STATE ===")
for k, v in result.items():
    if k != 'messages':
        print(f"{k}: {str(v)[:150]}")
print("\n=== MESSAGE TRACE ===")
for m in result['messages']:
    label = getattr(m, 'name', m.type)
    print(f"[{label}] {m.content[:200]}")
